# MScFE 660 · GWP3 — CrudeCortex: results, interpretation and improvement
Group 14660. Executable, offline notebook using the bundled FRED snapshot. **All numbers below are computed here**; recorded values from the submitted Colab notebook and verified Alvi (2018) figures are shown separately and labelled.

In [1]:
import sys, warnings; sys.path.insert(0, '..'); warnings.filterwarnings('ignore')
import pandas as pd; pd.set_option('display.width',180); pd.set_option('display.max_columns',30)
from core import gwp3_pipeline as P, gwp3_config as G, gwp3_charts as X
res = P.run(G.GWP3Config())

## 1. Training / validation / testing definitions (chronological 80/10/10)

In [2]:
d = P.describe_splits(res.prep); d.round(2).T

Split,Training,Validation,Testing
N,180,22,23
Share %,80.0,9.78,10.22
Start,2006-04-01,2021-04-01,2023-02-01
End,2021-03-01,2023-01-01,2024-12-01
WTI mean $,70.81,84.46,77.05
Ret mean %,-0.01,1.03,-0.47
Ret std %,11.6,8.77,6.13
Skew,-0.96,0.12,-0.05
Kurt,7.62,-1.11,-1.21
Ann. vol %,40.2,30.4,21.23


## 2. Validation vs testing comparison

In [3]:
t = res.tests; {k: (round(v,4) if isinstance(v,float) else v) for k,v in t.items() if k!='acf'}

{'ks_stat': 0.1897,
 'ks_p': 0.7362,
 'mw_stat': 273.0,
 'mw_p': 0.6579,
 'contingency': [[11, 11, 0], [3, 20, 0]],
 'regime_test': 'Fisher exact (2 occupied regimes)',
 'regime_p': 0.0106,
 'acf_ci': 0.1307}

## 3. Scoring-function selection (validation only) and replication of Alvi §4.3 in spirit

In [4]:
res.candidates

,n_edges,target_parents,val_accuracy,val_log_loss,fallbacks
score,,,,,
bic,1,1,0.454545,0.899780,0
k2,4,0,0.409091,0.817899,0
bdeu,1,1,0.454545,0.899780,0


In [5]:
print('Selected:', res.selected, '| edges:', res.edges)
print(G.REPLICATION_VERDICT)
import pandas as pd
pd.DataFrame({'Alvi (verified)':[G.ALVI_VERIFIED['val_accuracy_pct']/100, G.ALVI_VERIFIED['test_accuracy_pct']/100],'Group notebook (recorded)':[G.RECORDED_NOTEBOOK['val_accuracy'],G.RECORDED_NOTEBOOK['test_accuracy']],'Computed here':[res.metrics['val']['accuracy'],res.metrics['test']['accuracy']]}, index=['validation','test']).round(3)

Selected: bic | edges: [('IndPro_Trend', 'WTI_Regime')]
Partial / conceptual replication, not like-for-like. Alvi (2018) uses a binary-difference discrete HMM per series, expert-seeded K2 hill climbing and a 28-observation test set; the group (and this app) use a Gaussian two-feature HMM on WTI plus tertile-discretised macro variables on a different sample. Accuracy figures are therefore not directly comparable.


,Alvi (verified),Group notebook (recorded),Computed here
validation,0.321,0.478,0.455
test,0.571,0.640,0.609


## 4. Forecast accuracy, baselines and graphs (test evaluated once)

In [6]:
pd.DataFrame(res.metrics).T.round(3)

,n,correct,accuracy,ci_low,ci_high,macro_f1,balanced_acc,log_loss,binom_p_vs_baseline,majority_acc,persistence_acc,random_acc_expected,mcnemar_p_vs_majority
val,22.0,10.0,0.455,0.269,0.653,0.436,0.455,0.900,0.738,0.50,0.909,0.552,1.000
test,23.0,14.0,0.609,0.408,0.778,0.517,0.633,0.541,1.000,0.87,0.913,0.552,0.109


In [7]:
for k in ('val','test'):
    print(k); print(res.confusion[k]); print(res.report[k].round(2))

val
                 pred Bear  pred Stagnant  pred Bull
actual Bear              3              8          0
actual Stagnant          4              7          0
actual Bull              0              0          0
          precision  recall    f1  support
class                                     
Bear           0.43    0.27  0.33       11
Stagnant       0.47    0.64  0.54       11
Bull            NaN     NaN   NaN        0
test
                 pred Bear  pred Stagnant  pred Bull
actual Bear              2              1          0
actual Stagnant          8             12          0
actual Bull              0              0          0
          precision  recall    f1  support
class                                     
Bear           0.20    0.67  0.31        3
Stagnant       0.92    0.60  0.73       20
Bull            NaN     NaN   NaN        0


In [8]:
X.fig_accuracy_vs_baselines(res.metrics['test'], 'Test accuracy vs baselines').show()
X.fig_actual_vs_pred(res.prep.panel.loc[res.prep.split.reindex(res.prep.panel.index)=='test','WTI_Regime'], res.pred['test'], 'Actual vs predicted (test)').show()

## 5. Justified improvements (selected on validation; test reported afterwards)

In [9]:
P.compare_variants().round(3)

,Score (val-selected),Val acc,Val majority,Test acc,Test majority,Test macro-F1,Test log-loss,Target parents
Variant,,,,,,,,
V0 · Same-month nowcast (notebook replica),bic,0.455,0.500,0.609,0.870,0.517,0.541,1
V1 · One-month-lagged evidence (true 1-step-ahead),k2,0.500,0.500,0.696,0.870,0.517,0.593,0
V2 · V0 + causal filtered labels,k2,0.545,0.455,0.696,0.826,0.517,0.612,0
V3 · V1 + previous-regime node,bic,0.909,0.500,0.913,0.870,0.808,0.254,1
V4 · V0 + expert-seeded hill climb (Alvi style),bic,0.455,0.500,0.609,0.870,0.517,0.541,1


## 6. Robustness

In [10]:
P.robustness_splits().round(3)

,val n,test n,val acc,val majority,test acc,test majority
train/val/test,,,,,,
60/20/20,45,45,0.689,0.667,0.511,0.622
70/15/15,33,34,0.545,0.545,0.676,0.676
80/10/10,22,23,0.455,0.500,0.609,0.870


In [11]:
P.robustness_seeds().round(3)

,BIC,agreement with selected
seed,,
0,815.871,0.747
1,815.871,0.747
2,733.634,1.000
3,733.634,1.000
4,819.138,0.693
5,733.634,1.000
6,733.634,1.000
7,733.634,1.000


## 7. Illustrative rule (not a strategy)

In [12]:
tr = P.trading_demo(res); {k:v for k,v in tr.items() if k!='series'}

{'strategy_cum': 0.07011644537425621,
 'buy_hold_cum': -0.10240655401945709,
 'ann_sharpe': 0.34709755829767575,
 'n_trades': 11,
 'n_months': 23}

## 8. Contributions assessment

In [13]:
from core.gwp3_contrib import CONTRIBUTIONS
pd.DataFrame(CONTRIBUTIONS, columns=['ID','Claim','Status','Evidence','Caveat']).set_index('ID')

,Claim,Status,Evidence,Caveat
ID,,,,
C1,Replace EGARCH-M views in Black-Litterman with...,Not testable here,No Black-Litterman allocation was built in GWP3.,Needs portfolio optimisation and a benchmark; ...
C2,Novel use of HMM-discretised series as inputs ...,Replicated in spirit,The pipeline decodes HMM regimes and feeds the...,Novelty is Alvi's claim; we only show the cons...
C3,"Deployable, autonomous trading mechanism",Unsupported by our results,Our network does not beat the majority baselin...,"Costs, sizing and risk limits are absent; clai..."
C4,Structure learning needing no expert prior kno...,Replicated in spirit,Hill climbing runs with no seed (BIC/K2/BDeu) ...,"The learned graph is thin (few edges), so the ..."
C5,Python abstraction aids understanding of the d...,Design claim,The pipeline is a few hundred lines on open-so...,Subjective; not an empirical result.
C6,Systematic event-driven global-macro strategy ...,Not testable here,No fund-level comparison was run.,Comparative-return claim needs return data we ...
C7,Better models of energy markets for policy makers,Not testable here,A 5-node monthly model is a proof of concept.,"Policy use needs a larger, validated model."
C8,Amalgamates multi-disciplinary research to rai...,Unsupported by our results,No alpha is demonstrated; baseline not beaten.,Aspirational; the evidence here does not show ...


## 9. Discussion
On the test block the network does not beat the majority-class share; the submitted notebook shows the same pattern. Gains from a previous-regime node mainly reflect regime persistence. No claim of being better, faster or cheaper is made. Not investment advice.